# Notebook 01: Exploratory Data Analysis (EDA)
## Comparative Analysis of Machine Learning Algorithms for Air Quality Classification Across Multiple Datasets

**Objectives:**
1. Inspect schemas, column types, sample records, and date ranges across all 4 datasets (India, Italy, Beijing, Seoul).
2. Quantify missing value percentages and examine sentinel values (e.g. -200 in UCI Italy).
3. Analyze target category distributions and identify class imbalance.
4. Compute feature correlation matrices and identify multicollinearity.

In [ ]:
import os
import sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to sys.path
sys.path.insert(0, os.path.abspath('..'))
from src.data_loader import (
    load_india_dataset,
    load_italy_dataset,
    load_beijing_dataset,
    load_seoul_dataset
)
from src.visualization import (
    plot_missing_values,
    plot_class_distribution,
    plot_correlation_heatmap
)

print('Libraries successfully imported.')

### 1. Load and Inspect the Four Datasets

In [ ]:
df_india = load_india_dataset('../data/raw/india_city_day.csv')
df_italy = load_italy_dataset('../data/raw/italy_air_quality_uci.csv')
df_beijing = load_beijing_dataset('../data/raw/beijing_pm25.csv', aggregate_daily=True)
df_seoul = load_seoul_dataset('../data/raw/seoul_air_pollution.csv')

datasets = {
    'India (city_day)': df_india,
    'Italy (UCI AirQuality)': df_italy,
    'Beijing (UCI PM2.5)': df_beijing,
    'Seoul (Air Pollution)': df_seoul
}

for name, df in datasets.items():
    print(f'=== {name} ===')
    print(f'Shape: {df.shape}')
    target_col = 'AQI_Bucket' if 'AQI_Bucket' in df.columns else 'AQI_Category'
    print(f'Target: {target_col}')
    print(f'Missing total cells: {df.isnull().sum().sum()} ({(df.isnull().sum().sum() / df.size)*100:.2f}%)')
    print(df.head(2))
    print('-'*60)

### 2. Missing Value Analysis
Visualize missing feature percentages across each dataset.

In [ ]:
for name, df in datasets.items():
    missing = (df.isnull().sum() / len(df)) * 100
    missing = missing[missing > 0].sort_values(ascending=False)
    if not missing.empty:
        plt.figure(figsize=(9, 4))
        sns.barplot(x=missing.values, y=missing.index, palette='viridis')
        plt.title(f'Missing Value Percentage (%) — {name}')
        plt.xlabel('Percentage Missing')
        plt.show()

### 3. Target Class Distributions & Imbalance
Evaluate class frequencies for the six CPCB categories.

In [ ]:
for name, df in datasets.items():
    target_col = 'AQI_Bucket' if 'AQI_Bucket' in df.columns else 'AQI_Category'
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x=target_col, palette='rocket', order=['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe'])
    plt.title(f'Class Distribution — {name}')
    plt.xticks(rotation=15)
    plt.show()